<a href="https://colab.research.google.com/github/sumalya41/QFin.Colab/blob/main/BlackLitterman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 96.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
from scipy.optimize import minimize
import requests

In [ ]:
# =========================
# USER INPUT REQUIRED
# =========================
ASSETS = ["HDFCBANK.NS", "RELIANCE.NS", "ICICIBANK.NS", "BHARTIARTL.NS", "INFY.NS", "LT.NS", "SBIN.NS", "AXISBANK.NS", "ITC.NS" ]  # <-- CHANGE ASSETS
ASSETS = [asset.strip() for asset in ASSETS] # Clean asset names to remove any leading/trailing whitespace, including tabs
START_DATE = "2023-01-01"
END_DATE = "2026-01-01"
BASE_CURRENCY = "USD"
TARGET_CURRENCY = "INR"  # Example: "MXN"
INITIAL_INVESTMENT = 10000

# Black-Litterman Views
VIEWS = np.array([0.02, 0.01, 0.015, 0.025, 0.012, 0.018, 0.022, 0.017, 0.013])  # expected returns adjustments, updated to 9 elements
CONFIDENCE = 0.05

In [ ]:
# =========================
# 📂 DATA DOWNLOAD
# =========================
@st.cache_data
def load_data():
    data = yf.download(ASSETS, start=START_DATE, end=END_DATE)["Close"]
    return data.dropna()

data = load_data()

# =========================
# CURRENCY CONVERSION
# =========================
def convert_currency(data):
    if BASE_CURRENCY == TARGET_CURRENCY:
        return data

    fx_ticker = f"{BASE_CURRENCY}{TARGET_CURRENCY}=X"
    fx_data = yf.download(fx_ticker, start=START_DATE, end=END_DATE)

    if fx_data is None or fx_data.empty or "Close" not in fx_data.columns:
        print(f"Warning: Could not download or find 'Close' column for currency pair {fx_ticker}. Skipping currency conversion.")
        return data

    fx = fx_data["Close"]
    # Ensure fx is a Series, even if it's empty, before reindex and fillna
    if not isinstance(fx, pd.Series):
        print(f"Warning: 'Close' data for {fx_ticker} is not a pandas Series. Skipping currency conversion.")
        return data

    fx = fx.reindex(data.index).fillna(method="ffill")

    return data.mul(fx, axis=0)

data = convert_currency(data)

2026-04-29 06:48:46.946 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
/tmp/ipykernel_5299/3705173267.py:19: FutureWarning: YF.download() has changed argument auto_adjust default to True
  fx_data = yf.download(fx_ticker, start=START_DATE, end=END_DATE)
[*********************100%***********************]  1 of 1 completed

In [ ]:
# =========================
# 📈 RETURNS
# =========================
returns = data.pct_change().dropna()


In [ ]:
# =========================
# 💲 RISK-FREE RATE (FRED)
# =========================
def get_risk_free_rate():
    # USER INPUT REQUIRED
    API_KEY = "a7888afa471ffb5a8694b2014f894484"

    url = f"https://api.stlouisfed.org/fred/series/observations?series_id=DGS10&api_key={API_KEY}&file_type=json"
    response = requests.get(url).json()

    rates = [float(x["value"]) for x in response["observations"] if x["value"] != "."]
    return np.mean(rates) / 100

rf_rate = get_risk_free_rate()


In [ ]:
rf_rate

np.float64(0.05812361180278884)

In [ ]:
# =========================
# 📏 METRICS
# =========================
def portfolio_performance(weights):
    ret = np.sum(returns.mean() * weights) * 252
    vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * 252, weights)))
    return ret, vol

def sharpe_ratio(weights):
    ret, vol = portfolio_performance(weights)
    return (ret - rf_rate) / vol

def sortino_ratio(weights):
    downside = returns[returns < 0].std() * np.sqrt(252)
    ret, _ = portfolio_performance(weights)
    return (ret - rf_rate) / downside.mean()

def var_95(weights):
    portfolio_returns = returns.dot(weights)
    return np.percentile(portfolio_returns, 5)

def kurtosis(weights):
    portfolio_returns = returns.dot(weights)
    return portfolio_returns.kurtosis()


In [ ]:
# =========================
# 💪 OPTIMIZATION
# =========================
def optimize_portfolio(objective="sharpe"):
    num_assets = len(ASSETS)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    init_guess = num_assets * [1. / num_assets]

    if objective == "sharpe":
        func = lambda x: -sharpe_ratio(x)
    elif objective == "volatility":
        func = lambda x: portfolio_performance(x)[1]

    result = minimize(func, init_guess, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    return result.x

weights_sharpe = optimize_portfolio("sharpe")
weights_min_vol = optimize_portfolio("volatility")


In [ ]:
# =========================
# 🤯 BLACK-LITTERMAN
# =========================
def black_litterman():
    cov = returns.cov() * 252
    pi = returns.mean() * 252

    adjusted_returns = pi + CONFIDENCE * (VIEWS - pi)
    return adjusted_returns

bl_returns = black_litterman()

# =========================
# 📉 BACKTESTING
# =========================
def backtest(weights):
    portfolio = (returns.dot(weights) + 1).cumprod()
    sp500 = yf.download("^GSPC", start=START_DATE, end=END_DATE)["Close"]
    sp500 = sp500.pct_change().dropna()
    sp500 = (sp500 + 1).cumprod()

    return portfolio, sp500

portfolio_bt, sp500_bt = backtest(weights_sharpe)



/tmp/ipykernel_5299/2987170531.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download("^GSPC", start=START_DATE, end=END_DATE)["Close"]
[*********************100%***********************]  1 of 1 completed


In [ ]:
# efficient frontier
def efficient_frontier(points=100):
    num_assets = len(ASSETS)
    results = np.zeros((3, points))
    weights_record = []

    for i in range(points):
        weights = np.random.random(num_assets)
        weights /= np.sum(weights)
        weights_record.append(weights)

        ret, vol = portfolio_performance(weights)
        results[0, i] = vol
        results[1, i] = ret
        results[2, i] = (ret - rf_rate) / vol  # Sharpe

    return results, weights_record


In [ ]:
# MONTE CARLO SIMULATION
def monte_carlo_simulation(weights, simulations=1000, days=252):
    portfolio_returns = returns.dot(weights)
    mean = portfolio_returns.mean()
    std = portfolio_returns.std()

    simulations_data = []

    for _ in range(simulations):
        sim_returns = np.random.normal(mean, std, days)
        price = (1 + sim_returns).cumprod()
        simulations_data.append(price)

    return np.array(simulations_data)


In [ ]:
# =========================
# 📊 STREAMLIT UI
# =========================
st.title("📊 Portfolio Optimization Dashboard")

st.subheader("📈 Price Chart")
st.line_chart(data)

st.subheader("📊 Returns Heatmap")
sns.heatmap(returns.corr(), annot=True)
st.pyplot(plt)

st.subheader("💪 Optimized Portfolio (Max Sharpe)")
st.write(dict(zip(ASSETS, weights_sharpe)))

st.subheader("📉 Backtesting vs S&P 500")
bt_df = pd.DataFrame({
    "Portfolio": portfolio_bt,
    "S&P 500": sp500_bt.squeeze() # Convert DataFrame to Series
})
st.line_chart(bt_df)

st.subheader("📏 Risk Metrics")
st.write({
    "Sharpe": sharpe_ratio(weights_sharpe),
    "Sortino": sortino_ratio(weights_sharpe),
    "VaR (95%)": var_95(weights_sharpe),
    "Kurtosis": kurtosis(weights_sharpe)
})

st.subheader("🤯 Black-Litterman Adjusted Returns")
st.write(bl_returns)
# efficient frontier
st.subheader("🎯 Efficient Frontier")

results, weights_record = efficient_frontier()

plt.figure(figsize=(10,6))
plt.scatter(results[0,:], results[1,:], c=results[2,:], cmap='viridis')

# Highlight max Sharpe
max_sharpe_idx = np.argmax(results[2])
plt.scatter(results[0, max_sharpe_idx],
            results[1, max_sharpe_idx],
            color='red', s=100, label='Max Sharpe')

# Highlight min volatility
min_vol_idx = np.argmin(results[0])
plt.scatter(results[0, min_vol_idx],
            results[1, min_vol_idx],
            color='blue', s=100, label='Min Volatility')

plt.xlabel("Volatility")
plt.ylabel("Returns")
plt.colorbar(label="Sharpe Ratio")
plt.legend()

st.pyplot(plt)

# monte carlo
st.subheader("🎲 Monte Carlo Simulation")

mc_sim = monte_carlo_simulation(weights_sharpe, simulations=200)

plt.figure(figsize=(10,6))
for i in range(50):  # plot subset for clarity
    plt.plot(mc_sim[i], alpha=0.3)

plt.title("Monte Carlo Simulated Portfolio Paths")
plt.xlabel("Days")
plt.ylabel("Growth")

st.pyplot(plt)

# backtesting
st.subheader("📉 Backtesting vs Nifty50")

plt.figure(figsize=(10,6))
plt.plot(portfolio_bt, label="Portfolio")
plt.plot(sp500_bt, label="NIfty50")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Cumulative Returns")

st.pyplot(plt)

st.subheader("🤯 Black-Litterman Adjusted Returns")

bl_df = pd.DataFrame({
    "Asset": ASSETS,
    "Adjusted Returns": bl_returns
})

plt.figure(figsize=(8,5))
plt.bar(bl_df["Asset"], bl_df["Adjusted Returns"])
plt.xlabel("Assets")
plt.ylabel("Expected Returns")

st.pyplot(plt)

st.dataframe(bl_df)


2026-04-29 07:52:30.723 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 07:52:30.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 07:52:30.727 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 07:52:30.730 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 07:52:30.733 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 07:52:30.734 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 07:52:30.829 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 07:52:30.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

# =========================
# 📂 DATA DOWNLOAD
# =========================
@st.cache_data
def load_data():
    data = yf.download(ASSETS, start=START_DATE, end=END_DATE)["Close"]
    return data.dropna()

data = load_data()


# =========================
# 💱 CURRENCY CONVERSION
# =========================
def convert_currency(data):
    if BASE_CURRENCY == TARGET_CURRENCY:
        return data

    fx_ticker = f"{BASE_CURRENCY}{TARGET_CURRENCY}=X"
    fx_data = yf.download(fx_ticker, start=START_DATE, end=END_DATE)

    if fx_data is None or fx_data.empty or "Close" not in fx_data.columns:
        print(f"Warning: Could not download or find 'Close' column for currency pair {fx_ticker}. Skipping currency conversion.")
        return data

    fx = fx_data["Close"]
    # Ensure fx is a Series, even if it's empty, before reindex and fillna
    if not isinstance(fx, pd.Series):
        print(f"Warning: 'Close' data for {fx_ticker} is not a pandas Series. Skipping currency conversion.")
        return data

    fx = fx.reindex(data.index).fillna(method="ffill")

    return data.mul(fx, axis=0)

data = convert_currency(data)


# =========================
# 📈 RETURNS
# =========================
returns = data.pct_change().dropna()

# =========================
# 💲 RISK-FREE RATE (FRED)
# =========================
def get_risk_free_rate():
    # USER INPUT REQUIRED
    API_KEY = "YOUR_FRED_API_KEY"

    url = f"https://api.stlouisfed.org/fred/series/observations?series_id=DGS10&api_key={API_KEY}&file_type=json"
    response = requests.get(url).json()

    rates = [float(x["value"]) for x in response["observations"] if x["value"] != "."]
    return np.mean(rates) / 100

rf_rate = get_risk_free_rate()

# =========================
# 📏 METRICS
# =========================
def portfolio_performance(weights):
    ret = np.sum(returns.mean() * weights) * 252
    vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() * 252, weights)))
    return ret, vol

def sharpe_ratio(weights):
    ret, vol = portfolio_performance(weights)
    return (ret - rf_rate) / vol

def sortino_ratio(weights):
    downside = returns[returns < 0].std() * np.sqrt(252)
    ret, _ = portfolio_performance(weights)
    return (ret - rf_rate) / downside.mean()

def var_95(weights):
    portfolio_returns = returns.dot(weights)
    return np.percentile(portfolio_returns, 5)

def kurtosis(weights):
    portfolio_returns = returns.dot(weights)
    return portfolio_returns.kurtosis()

# =========================
# 💪 OPTIMIZATION
# =========================
def optimize_portfolio(objective="sharpe"):
    num_assets = len(ASSETS)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(num_assets))
    init_guess = num_assets * [1. / num_assets]

    if objective == "sharpe":
        func = lambda x: -sharpe_ratio(x)
    elif objective == "volatility":
        func = lambda x: portfolio_performance(x)[1]

    result = minimize(func, init_guess, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    return result.x

weights_sharpe = optimize_portfolio("sharpe")
weights_min_vol = optimize_portfolio("volatility")

# =========================
# 🤯 BLACK-LITTERMAN
# =========================
def black_litterman():
    cov = returns.cov() * 252
    pi = returns.mean() * 252

    adjusted_returns = pi + CONFIDENCE * (VIEWS - pi)
    return adjusted_returns

bl_returns = black_litterman()

# =========================
# 📉 BACKTESTING
# =========================
def backtest(weights):
    portfolio = (returns.dot(weights) + 1).cumprod()
    sp500 = yf.download("^GSPC", start=START_DATE, end=END_DATE)["Close"]
    sp500 = sp500.pct_change().dropna()
    sp500 = (sp500 + 1).cumprod()

    return portfolio, sp500

portfolio_bt, sp500_bt = backtest(weights_sharpe)

# =========================
# 📊 STREAMLIT UI
# =========================
st.title("📊 Portfolio Optimization Dashboard")

st.subheader("📈 Price Chart")
st.line_chart(data)

st.subheader("📊 Returns Heatmap")
sns.heatmap(returns.corr(), annot=True)
st.pyplot(plt)

st.subheader("💪 Optimized Portfolio (Max Sharpe)")
st.write(dict(zip(ASSETS, weights_sharpe)))

st.subheader("📉 Backtesting vs S&P 500")
bt_df = pd.DataFrame({
    "Portfolio": portfolio_bt,
    "S&P 500": sp500_bt
})
st.line_chart(bt_df)

st.subheader("📏 Risk Metrics")
st.write({
    "Sharpe": sharpe_ratio(weights_sharpe),
    "Sortino": sortino_ratio(weights_sharpe),
    "VaR (95%)": var_95(weights_sharpe),
    "Kurtosis": kurtosis(weights_sharpe)
})

st.subheader("🤯 Black-Litterman Adjusted Returns")
st.write(bl_returns)'''

In [ ]:

'''


